# Entrenamiento en Google Colab — Clasificación de Madurez de Paltas Hass

Este notebook ejecuta todo el pipeline en **Google Colab con GPU T4 gratuita**:
1. Verifica GPU CUDA disponible.
2. Carga el proyecto (desde GitHub, Google Drive o un `.zip` local subido).
3. Descarga automáticamente el dataset oficial desde **Mendeley Data** (~418 MB) y organiza las 14.710 imágenes.
4. Construye manifiesto, splits estratificados por fruta, figuras EDA y caché de 256×256 px.
5. Entrena los modelos elegidos (por defecto los modelos con **pérdida ordinal asimétrica**, que penaliza 2× clasificar maduras como inmaduras).
6. Evalúa asimetría, compara modelos, regenera figuras y presentaciones `.tex`, y empaqueta todo en `resultados_paltas_colab.zip`.

> **Antes de empezar:** En el menú superior ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Acelerador de hardware: GPU (T4)**.

In [1]:
# @title 1. Verificar GPU y entorno de Colab
import sys, torch

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__} | CUDA disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
else:
    raise RuntimeError(
        "No se detectó GPU. Activa GPU T4 en: Entorno de ejecución -> Cambiar tipo de entorno -> T4 GPU"
    )

Python: 3.13.16
PyTorch: 2.14.1+cu130 | CUDA disponible: True
GPU: Tesla T4
Tesla T4, 15360 MiB, 580.82.07


In [2]:
# @title 2. Obtener el código del proyecto
# Opciones para cargar el código:
#   - "github": clona desde REPO_URL y rama BRANCH (ideal cuando ya subiste tus cambios a tu fork/repo).
#   - "drive": copia o trabaja sobre una carpeta ya subida a Google Drive (ej. MyDrive/PALTAS).
#   - "upload_zip": te pide subir un .zip de tu carpeta local PALTAS (útil si aún no haces push a GitHub).
SOURCE_MODE = "github"  # @param ["github", "drive", "upload_zip"]
REPO_URL = "https://github.com/cmartinez1524/clasificacion-de-paltas.git"  # @param {type:"string"}
BRANCH = "main"  # @param {type:"string"}
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/PALTAS"  # @param {type:"string"}

# Asegurar que el entorno de ejecución está parado sobre un directorio válido existente antes de operar
%cd /content

import os, shutil, zipfile
from pathlib import Path

WORKDIR = Path("/content/clasificacion-de-paltas")

if SOURCE_MODE == "github":
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {WORKDIR}
elif SOURCE_MODE == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    shutil.copytree(
        DRIVE_PROJECT_PATH,
        WORKDIR,
        ignore=shutil.ignore_patterns(".venv", "__pycache__", ".git", "data/cache", "Hass Avocado*"),
    )
elif SOURCE_MODE == "upload_zip":
    from google.colab import files
    print("Sube un .zip con el contenido de tu carpeta PALTAS...")
    uploaded = files.upload()
    zip_name = next(iter(uploaded))
    if WORKDIR.exists():
        shutil.rmtree(WORKDIR)
    WORKDIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_name, "r") as zf:
        zf.extractall(WORKDIR)
    # Si el zip venía envuelto en una sola subcarpeta (ej. PALTAS/scripts/...), subir un nivel
    if not (WORKDIR / "scripts").exists():
        subfolders = [p for p in WORKDIR.iterdir() if p.is_dir() and (p / "scripts").exists()]
        if subfolders:
            for item in subfolders[0].iterdir():
                shutil.move(str(item), str(WORKDIR / item.name))

os.chdir(WORKDIR)
print(f"Directorio activo: {Path.cwd()}")
if not (WORKDIR / "configs" / "resnet50_asimetrico.yaml").exists():
    print(
        "[AVISO] No se encontró configs/resnet50_asimetrico.yaml en esta rama/copia. "
        "Asegúrate de apuntar REPO_URL/BRANCH a la rama con los cambios asimétricos, o usa drive/upload_zip."
    )
else:
    print("Configuraciones asimétricas detectadas correctamente.")

Cloning into '/content/clasificacion-de-paltas'...
remote: Enumerating objects: 115, done.
remote: Counting objects: 100% (115/115), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 115 (delta 17), reused 79 (delta 9), pack-reused 0 (from 0)
Receiving objects: 100% (115/115), 5.94 MiB | 27.65 MiB/s, done.
Resolving deltas: 100% (17/17), done.
Directorio activo: /content/clasificacion-de-paltas
Configuraciones asimétricas detectadas correctamente.


In [3]:
# @title 3. Instalar dependencias faltantes (conservando PyTorch + CUDA de Colab)
!pip install -q timm==1.0.19 openpyxl PyYAML tqdm scikit-learn matplotlib seaborn pandas gradio

import timm, pandas, sklearn, yaml
print(f"timm={timm.__version__} | pandas={pandas.__version__} | scikit-learn={sklearn.__version__}")

/usr/local/lib/python3.13/dist-packages/torch/jit/_script.py:1643: FutureWarning: `torch.jit.interface` is deprecated. Please use `torch.compile` instead.
  warnings.warn(


timm=1.0.19 | pandas=2.2.3 | scikit-learn=1.6.1


In [4]:
# @title 4. Descargar y organizar el dataset oficial desde Mendeley Data (~418 MB)
import shutil, urllib.request, zipfile
from pathlib import Path
from tqdm.auto import tqdm

MENDELEY_ZIP_URL = "https://data.mendeley.com/public-api/zip/3xd9n945v8/download/1"
RAW_ROOT = Path("Hass Avocado Ripening Photographic Dataset")
IMAGES_TARGET = RAW_ROOT / "Avocado Ripening Dataset"
EXCEL_TARGET = RAW_ROOT / "Avocado Ripening Dataset.xlsx"

def dataset_listo() -> bool:
    return EXCEL_TARGET.exists() and IMAGES_TARGET.exists() and len(list(IMAGES_TARGET.glob("*.jpg"))) >= 14710

if dataset_listo():
    print(f"Dataset ya presente: {len(list(IMAGES_TARGET.glob('*.jpg')))} imágenes.")
else:
    zip_path = Path("/content/mendeley_paltas.zip")
    tmp_dir = Path("/content/_tmp_mendeley")
    if tmp_dir.exists():
        shutil.rmtree(tmp_dir)
    tmp_dir.mkdir(parents=True, exist_ok=True)

    print("Descargando desde Mendeley Data...")
    req = urllib.request.Request(MENDELEY_ZIP_URL, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp, open(zip_path, "wb") as out:
        total = int(resp.headers.get("Content-Length", 0) or 0)
        with tqdm(total=total, unit="B", unit_scale=True, desc="Mendeley ZIP") as pbar:
            while True:
                chunk = resp.read(1024 * 1024)
                if not chunk:
                    break
                out.write(chunk)
                pbar.update(len(chunk))

    print("Descomprimiendo archivo principal...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(tmp_dir)

    # Descomprimir ZIPs anidados si Mendeley empaquetó sub-archivos comprimidos
    for inner_zip in list(tmp_dir.rglob("*.zip")):
        print(f"Descomprimiendo archivo interno: {inner_zip.name}")
        with zipfile.ZipFile(inner_zip, "r") as zf:
            zf.extractall(inner_zip.parent)

    RAW_ROOT.mkdir(parents=True, exist_ok=True)
    IMAGES_TARGET.mkdir(parents=True, exist_ok=True)

    xlsx_files = list(tmp_dir.rglob("*.xlsx"))
    assert xlsx_files, "No se encontró el archivo .xlsx dentro del ZIP descargado."
    shutil.move(str(xlsx_files[0]), str(EXCEL_TARGET))

    jpg_files = list(tmp_dir.rglob("*.jpg"))
    print(f"Organizando {len(jpg_files)} imágenes .jpg...")
    for jpg in jpg_files:
        dst = IMAGES_TARGET / jpg.name
        if not dst.exists():
            shutil.move(str(jpg), str(dst))

    shutil.rmtree(tmp_dir, ignore_errors=True)
    zip_path.unlink(missing_ok=True)

n_jpg = len(list(IMAGES_TARGET.glob("*.jpg")))
print(f"Verificación -> Excel: {EXCEL_TARGET.exists()} | Imágenes: {n_jpg}")
assert EXCEL_TARGET.exists() and n_jpg == 14710, f"Esperaba 14710 imágenes, encontré {n_jpg}"

Descargando desde Mendeley Data...


Mendeley ZIP:   0%|          | 0.00/418M [00:00<?, ?B/s]

Descomprimiendo archivo principal...
Organizando 14710 imágenes .jpg...
Verificación -> Excel: True | Imágenes: 14710


In [5]:
# @title 5. Preparar manifiesto, particiones, caché 256 px y EDA
!python scripts/00_build_manifest.py
!python scripts/01_make_splits.py
!python scripts/03_cache_images.py --workers 2
!python scripts/02_eda.py

Filas en Excel          : 14722
Imagenes en disco       : 14710
Descartadas (sin imagen): 12
  -> Tam_d02_052_a_2, Tam_d02_052_b_2, T10_d04_065_a_3, T10_d04_065_b_3, T10_d02_072_a_1, T10_d02_072_b_1, T20_d02_141_a_2, T20_d02_141_b_2, T20_d03_192_a_1, T20_d03_192_b_1, T20_d03_212_a_2, T20_d03_212_b_2
Filas en el manifiesto  : 14710

Integridad nombre-archivo vs Excel (0 = consistente):
  indice de madurez : 0
  grupo             : 0
  dia               : 0
  muestra           : 0

Frutas (muestras) unicas: 478
               imagenes  frutas
storage_group                  
T10                8866     192
T20                2920     143
Tam                2924     143

Distribucion de clases:
ripening_index
1    3568
2    2228
3    2756
4    3294
5    2864

Manifiesto escrito en: /content/clasificacion-de-paltas/data/manifest.csv
=== Particion agrupada por fruta (seed 42) ===
       imagenes  frutas  % imagenes  % frutas
split                                        
train     10208     3

In [6]:
# @title 6. Configurar y ejecutar el entrenamiento en GPU
# Puedes combinar o elegir los experimentos que quieras entrenar:
#   - "resnet50_asimetrico", "vit_small_asimetrico" (con pérdida ordinal asimétrica 2x)
#   - "resnet50", "vit_small" (baselines simétricos equiparables)
#   - "resnet50_scratch", "vit_small_scratch" (ablación sin pre-entrenamiento)
#   - "hybrid_fusion" (requiere resnet50.pt y vit_small.pt), "hybrid_vit_r26"

MODELOS_A_ENTRENAR = [
    "resnet50_asimetrico",
    "vit_small_asimetrico",
]

# Opcional: coloca un número (ej. 2) para hacer una corrida rápida de prueba; None usa el YAML completo
EPOCHS_OVERRIDE = None  # @param {type:"raw"}
BATCH_SIZE_OVERRIDE = None  # @param {type:"raw"}
NUM_WORKERS = 2  # @param {type:"integer"}

import subprocess

for nombre in MODELOS_A_ENTRENAR:
    cmd = [
        "python", "scripts/train.py",
        "--config", f"configs/{nombre}.yaml",
        "--num-workers", str(NUM_WORKERS),
    ]
    if EPOCHS_OVERRIDE is not None:
        cmd += ["--epochs", str(EPOCHS_OVERRIDE)]
    if BATCH_SIZE_OVERRIDE is not None:
        cmd += ["--batch-size", str(BATCH_SIZE_OVERRIDE)]

    print(f"\n>>> Ejecutando: {' '.join(cmd)}")
    subprocess.run(cmd, check=True)



>>> Ejecutando: python scripts/train.py --config configs/resnet50_asimetrico.yaml --num-workers 2
Experimento : resnet50_asimetrico
Modelo      : resnet50  (pretrained=True)
Dispositivo : cuda  Tesla T4
Imagenes    : /content/clasificacion-de-paltas/data/cache
Notas       : ResNet-50 con perdida ordinal asimetrica (2x falso inmaduro) y decision Bayesiana
  train: 10.208 imagenes  334 frutas
  val  :  2.240 imagenes   72 frutas
  test :  2.262 imagenes   72 frutas
  params: 23.52 M (entrenables 23.52 M)  | 8.17 GFLOPs @ 224px
ep  1/15     93s  train_loss=1.8713 train_acc=0.4519  |  val accuracy=0.6085  macro_f1=0.5994  qwk=0.8769  mae=0.4286  asymmetric_mae=0.6522  underestimation_rate=0.1915  off_by_one=0.9643  <- mejor
ep  2/15     92s  train_loss=1.2550 train_acc=0.6676  |  val accuracy=0.6679  macro_f1=0.6646  qwk=0.9121  mae=0.3438  asymmetric_mae=0.4701  underestimation_rate=0.1179  off_by_one=0.9884  <- mejor
ep  3/15     91s  train_loss=1.1493 train_acc=0.7024  |  val accuracy=

In [7]:
# @title 7. Evaluar asimetría, comparar modelos y regenerar presentaciones
from pathlib import Path
import subprocess

!python scripts/10_evaluar_asimetria.py

# Comparar automáticamente todos los modelos que tengan *_test.json disponible
orden_candidatos = [
    "resnet50",
    "vit_small",
    "resnet50_asimetrico",
    "vit_small_asimetrico",
    "resnet50_scratch",
    "vit_small_scratch",
    "hybrid_fusion",
    "hybrid_vit_r26",
]
disponibles = [m for m in orden_candidatos if Path(f"reports/metrics/{m}_test.json").exists()]
print("Modelos disponibles para comparar:", disponibles)

if len(disponibles) >= 2:
    subprocess.run([
        "python", "scripts/compare_models.py",
        "--models", *disponibles,
        "--tag", "final",
    ], check=True)

if Path("checkpoints/resnet50.pt").exists() and Path("checkpoints/vit_small.pt").exists():
    !python scripts/04_interpretability.py --mode disagreement
    !python scripts/09_mejoras_inferencia.py

!python scripts/08_figuras_didacticas.py
!python scripts/07_deck_latex.py

=== Resumen de asimetria ordinal (2x falso inmaduro, 1x falso maduro) ===
              modelo    n  accuracy    qwk    mae  asymmetric_mae  underestimation_rate  overestimation_rate  off_by_one
            resnet50 2262    0.7458 0.9356 0.2617          0.3904                0.1264               0.1278      0.9925
           vit_small 2262    0.7224 0.9313 0.2825          0.4293                0.1450               0.1326      0.9951
 resnet50_asimetrico 2262    0.7224 0.9296 0.2865          0.3532                0.0654               0.2122      0.9912
vit_small_asimetrico 2262    0.7476 0.9379 0.2573          0.3563                0.0977               0.1547      0.9951
    resnet50_scratch 2262    0.6715 0.9152 0.3519          0.4523                0.0959               0.2325      0.9766
   vit_small_scratch 2262    0.6123 0.8891 0.4094          0.6583                0.2352               0.1525      0.9810
       hybrid_fusion 2262    0.7675 0.9410 0.2378          0.3727              

In [8]:
# @title 8. Empaquetar checkpoints, métricas, figuras y entregables para descargar
import shutil, zipfile
from pathlib import Path

SAVE_TO_DRIVE = False  # @param {type:"boolean"}
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/PALTAS_resultados"  # @param {type:"string"}

zip_out = Path("/content/resultados_paltas_colab.zip")
carpetas_incluir = ["checkpoints", "reports", "entregables", "data/manifest.csv", "data/splits.csv"]

with zipfile.ZipFile(zip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for ruta in carpetas_incluir:
        p = Path(ruta)
        if not p.exists():
            continue
        if p.is_file():
            zf.write(p)
        else:
            for archivo in p.rglob("*"):
                if archivo.is_file():
                    zf.write(archivo)

size_mb = zip_out.stat().st_size / (1024 * 1024)
print(f"Paquete generado: {zip_out} ({size_mb:.1f} MB)")

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    shutil.copy2(zip_out, Path(DRIVE_OUTPUT_DIR) / zip_out.name)
    print(f"Copiado a Google Drive en: {DRIVE_OUTPUT_DIR}/{zip_out.name}")

try:
    from google.colab import files
    files.download(str(zip_out))
except ImportError:
    print("Descarga manual disponible en:", zip_out)

Paquete generado: /content/resultados_paltas_colab.zip (170.9 MB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>